# Full-methodology replication — Gemma-4-E2B-it

Adds to what we had before:
1. **MRR** (mean reciprocal rank) instead of just top-k — matches Neel Nanda + Anthropic scoring
2. **Causal swap intervention** — for each eval item, ablate the concept direction from the residual and measure Δ log-prob of the correct answer. If the workspace is causally used, ablation should hurt
3. **Shuffled-corpus control** — fit a second lens on token-shuffled prompts. If our workspace claim is real, shuffled MRR should collapse
4. **Prompt-truncation ablation on sanity probes** — cut prompts before the referent; if 'spider' still emerges the lens is broken

Reuses the already-fitted Gemma-4-E2B-it lens (attached as kernel data source).

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/anthropics/jacobian-lens.git',
    '--upgrade', 'transformers>=5.5'])
import torch
print(torch.cuda.get_device_name(0), '| torch:', torch.__version__)

In [ ]:
import os, glob, json, pathlib, time, re, random
import torch, transformers, jlens
from datasets import load_dataset
from jlens.hooks import ActivationRecorder

# Find the fitted lens
cands = glob.glob('/kaggle/input/**/gemma-4-e2b-jlens.pt', recursive=True)
assert cands, 'lens file not found'
LENS_PATH = cands[0]
print('Lens:', LENS_PATH)

MODEL = 'google/gemma-4-E2B-it'
OUT_DIR = pathlib.Path('/kaggle/working')
SKIP_FIRST = 4
TARGET_LAYER = -2

# --- Load model + lens once, use throughout ---
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, attn_implementation='sdpa', low_cpu_mem_usage=True
).to('cuda')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL)
model = jlens.from_hf(hf_model, tokenizer, compile=True)
lens = jlens.JacobianLens.load(LENS_PATH)
print(f'n_layers={model.n_layers}  d_model={model.d_model}  lens.n_prompts={lens.n_prompts}')

# Load eval data
subprocess.run(['git','clone','-q','--depth=1','https://github.com/anthropics/jacobian-lens.git','/tmp/jl'])
EVALS_DIR = pathlib.Path('/tmp/jl/data/evaluations')

## Part 1 — MRR + top-k on all 6 evals

MRR = average of `1 / (best rank of correct token across all layers)`. This is what Neel Nanda and Anthropic report. Higher = better.

In [ ]:
def tokens_of(word):
    ids = []
    for v in {word, ' '+word, word.lower(), ' '+word.lower()}:
        e = tokenizer(v, add_special_tokens=False).input_ids
        if len(e) == 1:
            ids.append(e[0])
    return sorted(set(ids))

def best_rank(lens_logits, token_ids):
    if not token_ids: return None
    best = None
    for _, logits in lens_logits.items():
        order = logits[0].argsort(descending=True)
        rank_lookup = torch.empty_like(order)
        rank_lookup[order] = torch.arange(order.numel(), device=order.device)
        cand = rank_lookup[torch.tensor(token_ids, device=order.device)]
        r = int(cand.min().item()) + 1
        if best is None or r < best: best = r
    return best

def score_eval(eval_path):
    data = json.load(eval_path.open())
    items = data['items']
    rr_list, top1_list, top10_list = [], [], []
    for item in items:
        try:
            lens_logits, _, _ = lens.apply(model, item['prompt'], positions=[-1])
        except Exception:
            continue
        inters = item['intermediates'] if isinstance(item['intermediates'], list) else [item['intermediates']]
        rrs, t1s, t10s = [], [], []
        for inter in inters:
            key = inter if isinstance(inter, str) else (inter.get('synonyms', [inter.get('key','')])[0] if isinstance(inter, dict) else inter[0])
            tok_ids = tokens_of(str(key))
            r = best_rank(lens_logits, tok_ids)
            if r is None: continue
            rrs.append(1.0/r); t1s.append(1.0 if r<=1 else 0.0); t10s.append(1.0 if r<=10 else 0.0)
        if rrs:
            rr_list.append(sum(rrs)/len(rrs))
            top1_list.append(sum(t1s)/len(t1s))
            top10_list.append(sum(t10s)/len(t10s))
    n = len(rr_list)
    return {'n': n, 'MRR': sum(rr_list)/max(1,n), 'top1': sum(top1_list)/max(1,n), 'top10': sum(top10_list)/max(1,n)}

probing_results = {}
for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
    r = score_eval(path)
    probing_results[path.stem] = r
    print(f'  {path.stem:<30}  n={r["n"]:>3}  MRR={r["MRR"]:.4f}  top-1={r["top1"]:.4f}  top-10={r["top10"]:.4f}')

(OUT_DIR / 'probing_results.json').write_text(json.dumps(probing_results, indent=2))

## Part 2 — Causal swap intervention (Neel Nanda's actual test)

For each item: (a) get concept direction from the lens Jacobian; (b) hook a residual-modification into each workspace-band layer that ablates that direction; (c) measure the delta in log-probability of the correct final answer.

If the concept representation is *causally used* by the model, ablation should drop the answer probability. If the lens finds concepts that are just correlated but unused, no ablation effect.

In [ ]:
def unembed_direction_for_token(model_hf, token_id):
    """Return the unembedding row for a token — the direction the model uses to write that token."""
    W = model_hf.get_output_embeddings().weight  # [vocab, d]
    return W[token_id].detach().to('cuda').float()

def causal_swap_score(item, layers_to_ablate=None):
    """For one item, ablate concept-direction from the residual at given layers, measure delta log-prob of correct answer."""
    prompt = item['prompt']
    inters = item['intermediates'] if isinstance(item['intermediates'], list) else [item['intermediates']]
    target = item.get('target') or item.get('answer')
    if not target: return None
    if isinstance(target, list): target = target[0]
    target_ids = tokens_of(str(target))
    if not target_ids: return None

    # Get baseline logits at final position
    input_ids = model.encode(prompt, max_length=128)
    if input_ids.ndim == 1: input_ids = input_ids.unsqueeze(0)
    with torch.no_grad():
        out = hf_model(input_ids)
        logits = out.logits
        # shape [1, seq, vocab] — take last position
        baseline_logits = logits[0, -1]
        baseline_lp = torch.log_softmax(baseline_logits.float(), dim=-1)
    base_target_lp = max(float(baseline_lp[tid]) for tid in target_ids)

    # Build the concept direction from the FIRST intermediate that tokenizes cleanly
    concept_dir = None
    for inter in inters:
        key = inter if isinstance(inter, str) else (inter.get('synonyms', [inter.get('key','')])[0] if isinstance(inter, dict) else inter[0])
        tok_ids = tokens_of(str(key))
        if tok_ids:
            concept_dir = unembed_direction_for_token(hf_model, tok_ids[0])
            break
    if concept_dir is None: return None
    concept_dir = concept_dir / concept_dir.norm()

    # Ablate concept direction at each workspace-band layer
    if layers_to_ablate is None:
        # Middle third of layers (workspace band, approx)
        L = model.n_layers
        layers_to_ablate = list(range(L//3, 2*L//3))

    handles = []
    def make_hook():
        def hook(module, inputs, output):
            tensor = output if torch.is_tensor(output) else output[0]
            proj = (tensor.float() @ concept_dir) # scalar per position
            new = tensor - (proj.unsqueeze(-1) * concept_dir).to(tensor.dtype)
            if torch.is_tensor(output):
                return new
            return (new,) + tuple(output[1:])
        return hook

    for li in layers_to_ablate:
        handles.append(model.layers[li].register_forward_hook(make_hook()))

    try:
        with torch.no_grad():
            out = hf_model(input_ids)
            ablated_logits = out.logits[0, -1]
            ablated_lp = torch.log_softmax(ablated_logits.float(), dim=-1)
        ablated_target_lp = max(float(ablated_lp[tid]) for tid in target_ids)
    finally:
        for h in handles: h.remove()

    return {
        'base_lp': base_target_lp,
        'ablated_lp': ablated_target_lp,
        'delta_lp': ablated_target_lp - base_target_lp,   # negative = ablation hurt (concept is used)
    }

def causal_score_eval(eval_path, max_items=60):
    """Run causal swap on up to `max_items` items to keep runtime reasonable."""
    data = json.load(eval_path.open())
    items = data['items'][:max_items]
    deltas = []
    for item in items:
        r = causal_swap_score(item)
        if r is not None: deltas.append(r['delta_lp'])
    n = len(deltas)
    if n == 0: return {'n': 0}
    mean_delta = sum(deltas)/n
    n_helped = sum(1 for d in deltas if d < -0.1)  # concept ablation hurt answer by >0.1 nats
    return {'n': n, 'mean_delta_lp': mean_delta, 'frac_ablation_hurts': n_helped/n}

causal_results = {}
for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
    r = causal_score_eval(path)
    causal_results[path.stem] = r
    if r.get('n'):
        print(f"  {path.stem:<30}  n={r['n']:>3}  Δlp={r['mean_delta_lp']:+.4f}  ablation_hurts={r['frac_ablation_hurts']:.3f}")
    else:
        print(f"  {path.stem:<30}  no items scoreable")

(OUT_DIR / 'causal_results.json').write_text(json.dumps(causal_results, indent=2))

## Part 3 — Prompt-truncation ablation

Do the sanity-probe concepts still emerge in the lens readout when the prompt is truncated *before* the referent phrase? If so, the lens is projecting priors, not workspace-formed concepts.

In [ ]:
probes = [
    {'full': 'The number of legs on the animal that spins webs is', 'trunc': 'The number of legs on the', 'target': 'spider'},
    {'full': 'The capital of the country shaped like a boot is',    'trunc': 'The capital of the country',  'target': 'italy'},
    {'full': 'Fact: The currency used in the country whose flag has a red maple leaf is the', 'trunc': 'Fact: The currency used in the country', 'target': 'canada'},
]

def top5_at_all_layers(prompt):
    lens_logits, _, _ = lens.apply(model, prompt, positions=[-2])
    return {int(k): [tokenizer.decode([t]).strip() for t in v[0].topk(5).indices.tolist()] for k,v in lens_logits.items()}

truncation_out = {}
for p in probes:
    full_readouts = top5_at_all_layers(p['full'])
    trunc_readouts = top5_at_all_layers(p['trunc'])
    full_hits = [l for l,toks in full_readouts.items() if any(p['target'] in t.lower() for t in toks)]
    trunc_hits = [l for l,toks in trunc_readouts.items() if any(p['target'] in t.lower() for t in toks)]
    truncation_out[p['target']] = {
        'full_prompt_hit_layers':  full_hits,
        'trunc_prompt_hit_layers': trunc_hits,
        'full_readouts': full_readouts,
        'trunc_readouts': trunc_readouts,
    }
    print(f"\n  target={p['target']!r}:")
    print(f"    full  ({p['full']!r:.80s}...): {len(full_hits)} hits at layers {full_hits[:8]}")
    print(f"    trunc ({p['trunc']!r:.80s}...): {len(trunc_hits)} hits at layers {trunc_hits[:8]}")

(OUT_DIR / 'truncation_results.json').write_text(json.dumps(truncation_out, indent=2))

## Part 4 — Shuffled-corpus control

Fit a *second* lens on the same 25 pile prompts but with token positions permuted within each prompt. If our workspace claim is real, MRR should collapse toward zero on this shuffled lens.

In [ ]:
# Free existing compiled state and lens memory before fitting a new one
del lens
torch.cuda.empty_cache()

# Prepare shuffled prompts — same 25 pile-10k items, but internally token-scrambled
ds = load_dataset('NeelNanda/pile-10k', split='train')
pile_prompts = []
for row in ds:
    if isinstance(row['text'], str) and len(row['text']) > 200:
        pile_prompts.append(row['text'])
        if len(pile_prompts) >= 25: break

rng = random.Random(42)
shuffled_prompts = []
for p in pile_prompts:
    ids = tokenizer(p, add_special_tokens=False).input_ids[:128]
    rng.shuffle(ids)
    shuffled_prompts.append(tokenizer.decode(ids))

print('Fitting shuffled-corpus lens (this is the expensive step, ~1-2h)...')
t0 = time.time()
shuffled_lens = jlens.fit(
    model, prompts=shuffled_prompts,
    target_layer=TARGET_LAYER, dim_batch=2,
    max_seq_len=96, skip_first=SKIP_FIRST,
    checkpoint_path=str(OUT_DIR / 'shuffled_ckpt.pt'),
)
print(f'Shuffled fit done in {(time.time()-t0)/60:.1f} min')
shuffled_lens.save(str(OUT_DIR / 'gemma-shuffled-lens.pt'))

In [ ]:
# Re-score with the shuffled lens
lens = shuffled_lens
shuffled_probing = {}
for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
    r = score_eval(path)
    shuffled_probing[path.stem] = r
    print(f'  [shuffled] {path.stem:<30}  n={r["n"]:>3}  MRR={r["MRR"]:.4f}  top-10={r["top10"]:.4f}')

(OUT_DIR / 'shuffled_probing.json').write_text(json.dumps(shuffled_probing, indent=2))

# --- Summary comparison ---
summary = {}
for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
    key = path.stem
    real = probing_results.get(key, {})
    sh   = shuffled_probing.get(key, {})
    caus = causal_results.get(key, {})
    summary[key] = {
        'real':     {'MRR': real.get('MRR'), 'top10': real.get('top10'), 'n': real.get('n')},
        'shuffled': {'MRR': sh.get('MRR'),   'top10': sh.get('top10'),   'n': sh.get('n')},
        'causal':   caus,
    }
(OUT_DIR / 'full_replication_summary.json').write_text(json.dumps(summary, indent=2))
print('\n=== SUMMARY ===')
print(f'{"eval":<30} {"real MRR":>10} {"shuf MRR":>10} {"Δ (↑)":>10} {"Δlp caus":>10}')
for k, v in summary.items():
    real_mrr = v['real'].get('MRR') or 0.0
    shuf_mrr = v['shuffled'].get('MRR') or 0.0
    d = real_mrr - shuf_mrr
    caus = v['causal'].get('mean_delta_lp')
    caus_str = f'{caus:+.3f}' if caus is not None else '  n/a'
    print(f'{k:<30} {real_mrr:>10.4f} {shuf_mrr:>10.4f} {d:>+10.4f} {caus_str:>10}')

print('\nDone. All outputs in /kaggle/working/')
for f in OUT_DIR.iterdir():
    print(' ', f.name, f'{f.stat().st_size/1e6:.1f} MB')